In [29]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import GroupShuffleSplit

PATH_CLIPS                          = "../../Data/Videos/Clips/"
PATH_ACTIONS_FILTERED               = "../../Data/Processed/actions_filtered.csv"

PATH_KEYPOINTS                      = "../../Data/Unprocessed/keypoints.csv"
PATH_METRICS                        = "../../Data/Unprocessed/metrics.csv"
PATH_ROI                            = "../../Data/Unprocessed/roi.csv"

PATH_CLS_TRAIN                       = "../../Data/Processed/cls_train.csv"
PATH_CLS_TEST                        = "../../Data/Processed/cls_test.csv"

PATH_ROI_TRAIN                       = "../../Data/Processed/roi_train.csv"
PATH_ROI_TEST                        = "../../Data/Processed/roi_test.csv"

WINDOW_SIZE                         = 4
WINDOWS_PER_ACTION                  = 5
NUM_JOINTS                          = 17

In [30]:
df_metrics = pd.read_csv(PATH_METRICS)

total_expected_frames = df_metrics["expected"].sum()
total_actual_frames = df_metrics["actual"].sum()
total_coverage = total_actual_frames / total_expected_frames

print(df_metrics[df_metrics["expected"] != df_metrics["actual"]])
print("")

print("Expected frames: ", total_expected_frames)
print("Actual frames:   ", total_actual_frames)
print(f"Coverage:         {total_coverage*100:.2f}%")

    fencer  action_id  start_frame  end_frame  expected  actual   coverage  \
265  RIGHT        995           32         39         8       7  87.500000   
598  RIGHT        774           26         32         7       6  85.714286   

               file  
265  6/20_Right.mp4  
598   5/15_Left.mp4  

Expected frames:  16821
Actual frames:    16819
Coverage:         99.99%


In [31]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)

df_keypoints["keypoints"] = df_keypoints["keypoints"].apply(
    lambda x: [tuple(p) for p in ast.literal_eval(x)] if isinstance(x, str) else x
)

In [32]:
df_filtered = pd.read_csv(PATH_ACTIONS_FILTERED)

df_merged = df_keypoints.merge(df_filtered, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

df_merged = df_merged[["file", "fencer", "action_id", "action", "frame", "start_frame", "end_frame", "confidence", "keypoints"]].reset_index(drop=True)
print(df_merged)

                file fencer  action_id           action  frame  start_frame  \
0      1/10_Left.mp4   LEFT          0  OTHER_NO_ACTION      0            0   
1      1/10_Left.mp4   LEFT          0  OTHER_NO_ACTION      1            0   
2      1/10_Left.mp4   LEFT          0  OTHER_NO_ACTION      2            0   
3      1/10_Left.mp4   LEFT          0  OTHER_NO_ACTION      3            0   
4      1/10_Left.mp4   LEFT          0  OTHER_NO_ACTION      4            0   
...              ...    ...        ...              ...    ...          ...   
16816   6/9_Left.mp4  RIGHT       1107     ATTACK_BASIC     40           38   
16817   6/9_Left.mp4  RIGHT       1107     ATTACK_BASIC     41           38   
16818   6/9_Left.mp4  RIGHT       1107     ATTACK_BASIC     42           38   
16819   6/9_Left.mp4  RIGHT       1107     ATTACK_BASIC     43           38   
16820   6/9_Left.mp4  RIGHT       1107     ATTACK_BASIC     44           38   

       end_frame  confidence  \
0             22   

In [33]:
df_merged["duration"] = df_merged["end_frame"] - df_merged["start_frame"] + 1

# Apply adjustment
mask = (df_merged["action"] == "DEFENSE_DISTANCE_PULL") & (df_merged["duration"] >= 8)

print(len(df_merged[mask]))

df_merged.loc[mask, "end_frame"] = df_merged.loc[mask, "end_frame"] - 4
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

print(len(df_merged[mask]))

# Drop helper column if desired
df_merged = df_merged.drop(columns="duration")

875
659


/tmp/ipykernel_7596/149812164.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(len(df_merged[mask]))


In [34]:
gss = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df_merged, groups=df_merged['file'])
)

df_train = df_merged.iloc[train_idx]
df_test  = df_merged.iloc[test_idx]

In [49]:
action_counts = df_train.groupby("action")["action_id"].nunique().sort_values(ascending=False)
action_counts_dropped = action_counts.copy()
action_counts_dropped.drop("OTHER_NO_ACTION", inplace=True, errors="ignore")
max_count = action_counts_dropped.max()

class_weights = max_count / action_counts
class_weights["ATTACK_LUNGE"] = 4
class_weights["ATTACK_BASIC"] = 6.6
class_weights["DEFENSE_DISTANCE_PULL"] = 6.7
class_weights["DEFENSE_PARRY"] = 18
weights_dict = class_weights.to_dict()

print(action_counts)
print("")
print(class_weights)

action
OTHER_NO_ACTION          480
ATTACK_LUNGE             178
ATTACK_BASIC             129
DEFENSE_DISTANCE_PULL     53
DEFENSE_PARRY             22
Name: action_id, dtype: int64

action
OTHER_NO_ACTION           0.370833
ATTACK_LUNGE              4.000000
ATTACK_BASIC              6.600000
DEFENSE_DISTANCE_PULL     6.700000
DEFENSE_PARRY            18.000000
Name: action_id, dtype: float64


In [36]:
def create_action_windows(df, window_size=4, num_windows=4, class_weights=None, random_state=42):
    rng = np.random.default_rng(random_state)
    window_rows = []
    window_counter = 0

    # Sort for safety
    df = df.sort_values(["action_id", "frame"]).reset_index(drop=True)

    for _, group in df.groupby(["action_id"]):
        action = group["action"].iloc[0]

        # Determine number of windows for this action
        weight = class_weights.get(action, 1.0) if class_weights else 1.0
        windows_per_action = max(1, int(round(num_windows * weight)))

        frames = group["frame"].values
        max_start = len(frames) - window_size
        if max_start < 0:
            continue  # action too short for a single window

        start_indices = rng.choice(
            np.arange(0, max_start + 1),
            size=windows_per_action,
            replace=False if windows_per_action <= max_start + 1 else True
        )

        for start in start_indices:
            window_counter += 1
            window_id = window_counter

            window_slice = group.iloc[start:start + window_size].copy()
            window_slice["window_id"] = window_id
            window_rows.append(window_slice)

    return pd.concat(window_rows, ignore_index=True)

In [50]:
df_train_windowed = create_action_windows(df_train, window_size=WINDOW_SIZE, num_windows=WINDOWS_PER_ACTION, class_weights=weights_dict)
df_test_windowed  = create_action_windows(df_test, window_size=WINDOW_SIZE, num_windows=WINDOWS_PER_ACTION)

train_counts = df_train_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)
test_counts  = df_test_windowed.groupby("action")["window_id"].nunique().sort_values(ascending=False)

print(train_counts)
print("")
print(test_counts)

print("")
print("Number of training action snippets: ", df_train_windowed["window_id"].nunique())
print("Number of testing action snippets:  ", df_test_windowed["window_id"].nunique())

action
ATTACK_BASIC             3729
ATTACK_LUNGE             3460
DEFENSE_DISTANCE_PULL    1768
DEFENSE_PARRY            1620
OTHER_NO_ACTION           828
Name: window_id, dtype: int64

action
OTHER_NO_ACTION          600
ATTACK_LUNGE             250
ATTACK_BASIC             195
DEFENSE_DISTANCE_PULL     75
DEFENSE_PARRY             35
Name: window_id, dtype: int64

Number of training action snippets:  11405
Number of testing action snippets:   1155


In [38]:
def explode_keypoints(df):
    # Expand each tuple into separate x,y columns
    exploded = df["keypoints"].apply(
        lambda kp: [coord for point in kp for coord in point]
    )

    # Create column names: x0, y0, x1, y1, ...
    num_points = len(df.iloc[0]["keypoints"])
    cols = [f"x{i}" for i in range(num_points)] + [f"y{i}" for i in range(num_points)]
    
    new_df = pd.DataFrame(exploded.tolist(), columns=cols)
    return pd.concat([df.drop(columns=["keypoints"]), new_df], axis=1)

In [48]:
df_train_data = df_train_windowed[["file", "fencer", "window_id", "frame", "action", "keypoints"]].copy()
df_test_data = df_test_windowed[["file", "fencer", "window_id", "frame", "action", "keypoints"]].copy()

df_train_data.sort_values(["window_id", "frame"], inplace=True)
df_test_data.sort_values(["window_id", "frame"], inplace=True)

df_train_data = explode_keypoints(df_train_data)
df_test_data = explode_keypoints(df_test_data)

df_train_data.to_csv(PATH_CLS_TRAIN, index=False)
df_test_data.to_csv(PATH_CLS_TEST, index=False)

In [40]:
df_roi = pd.read_csv(PATH_ROI)

gss_roi = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    random_state=42
)

train_idx, test_idx = next(
    gss_roi.split(df_roi, groups=df_roi['file'])
)

df_train_roi = df_roi.iloc[train_idx]
df_test_roi  = df_roi.iloc[test_idx]

df_train_roi.to_csv(PATH_ROI_TRAIN, index=False)
df_test_roi.to_csv(PATH_ROI_TEST, index=False)